# 05 · Training

`NextSequence` owns the model, the loss, the optimizer and the training loop. You pass DataLoaders.

The first cell rebuilds the data from **02_data_preparation** in compact form.

In [1]:
import random
import pandas as pd

ITEMS = ["apple", "bread", "cheese", "dates", "eggs"]
PRICE = {"apple": 20.0, "bread": 45.0, "cheese": 180.0, "dates": 95.0, "eggs": 60.0}

def make_events(n_users=300, seed=0):
    """One row per purchase. Each shopper buys ITEMS in a fixed cycle, starting at a random item."""
    rng = random.Random(seed)
    rows = []
    for user in range(n_users):
        start = rng.randrange(len(ITEMS))
        for t in range(rng.randint(6, 12)):
            item = ITEMS[(start + t) % len(ITEMS)]
            rows.append({"user_id": user, "t": t, "item": item,
                         "price": round(PRICE[item] * rng.uniform(0.9, 1.1), 2),
                         "weekday": rng.randrange(7)})
    return pd.DataFrame(rows)

import math
import torch
from torch.utils.data import DataLoader
from next_sequence import (CoreDataset, CustomCollate, Feature, NextSequence,
                           NextSequenceConfig, TaskType, create_bow, make_collate)

events = make_events()
users = (events.sort_values(["user_id", "t"]).groupby("user_id").agg(list)
               .reset_index().drop(columns="t").to_dict(orient="records"))

def cut(user, end):
    return {k: (v[:end] if isinstance(v, list) else v) for k, v in user.items()}

train_raw = [cut(u, -2) for u in users]      # target: 3rd-to-last event
val_raw = [cut(u, -1) for u in users]        # target: 2nd-to-last event
test_raw = [cut(u, None) for u in users]     # target: last event

bow = create_bow(u["item"] for u in train_raw)
log_prices = [math.log1p(p) for u in train_raw for p in u["price"]]
price_mean = sum(log_prices) / len(log_prices)
price_std = (sum((v - price_mean) ** 2 for v in log_prices) / len(log_prices)) ** 0.5

def transform(samples):
    return [{"item": [bow.get(i, bow["UNK"]) for i in u["item"]],
             "price": [(math.log1p(p) - price_mean) / price_std for p in u["price"]],
             "weekday": [d + 1 for d in u["weekday"]]} for u in samples]

train, val, test = transform(train_raw), transform(val_raw), transform(test_raw)

features = [
    Feature("item", TaskType.CLASSIFICATION, vocab_size=len(bow)),
    Feature("price", TaskType.REGRESSION, is_linear=True),
    Feature("weekday", TaskType.PASSTHROUGH, vocab_size=8),
]
funcs = {"item": make_collate("long"), "price": make_collate("float"), "weekday": make_collate("long")}
config = NextSequenceConfig(features=features, max_len=16, emb_dim=32, attn_heads=2)

def loader(samples, inference=False, shuffle=False, batch_size=32):
    collate = CustomCollate.from_config(config, "item", funcs, inference=inference)
    return DataLoader(CoreDataset(samples), batch_size=batch_size, shuffle=shuffle, collate_fn=collate)

print(f"{len(users)} users | train/val/test samples: {len(train)}/{len(val)}/{len(test)} | vocab: {len(bow)}")

300 users | train/val/test samples: 300/300/300 | vocab: 7


## Training settings

| Argument | Default | Meaning |
|---|---|---|
| `epochs` | 10 | passes over the training loader |
| `lr` | 3e-4 | peak learning rate (AdamW, warmup then cosine decay) |
| `weight_decay` | 0.01 | on weight matrices only |
| `warmup_ratio` | 0.05 | share of steps used for warmup |
| `grad_clip` | 1.0 | max gradient norm (`None` to disable) |
| `device` | `None` | CUDA if available, then MPS, then CPU |
| `seed` | `None` | makes initialization and training reproducible |
| `warm_start` | `False` | a second `fit` continues instead of starting over |

In [2]:
model = NextSequence(config, epochs=15, lr=3e-3, seed=0)
model

NextSequence(features=['item', 'price', 'weekday'], epochs=15, lr=0.003, device='cuda', not fitted)

## `fit`

With a validation loader, the weights of the epoch with the **lowest validation loss** are restored
at the end. Loaders are checked first: one built with different settings than the model is rejected
before any training happens.

In [3]:
model.fit(loader(train, shuffle=True), loader(val))
print("best epoch:", model.best_epoch_)

epoch=1 | train_loss=2.4540 | train_item=1.7857 | train_price=1.3515 | val_loss=1.9034 | val_item=1.5804 | val_price=0.6857
epoch=2 | train_loss=1.5506 | train_item=1.3007 | train_price=0.5411 | val_loss=1.2984 | val_item=1.0942 | val_price=0.4572


epoch=3 | train_loss=0.9820 | train_item=0.8183 | train_price=0.3599 | val_loss=0.6405 | val_item=0.5516 | val_price=0.1947
epoch=4 | train_loss=0.4758 | train_item=0.3964 | train_price=0.1654 | val_loss=0.3478 | val_item=0.2663 | val_price=0.1653


epoch=5 | train_loss=0.2040 | train_item=0.1676 | train_price=0.0778 | val_loss=0.1084 | val_item=0.0889 | val_price=0.0534
epoch=6 | train_loss=0.0696 | train_item=0.0611 | train_price=0.0434 | val_loss=0.0273 | val_item=0.0360 | val_price=0.0283


epoch=7 | train_loss=0.0181 | train_item=0.0320 | train_price=0.0318 | val_loss=-0.0068 | val_item=0.0231 | val_price=0.0203
epoch=8 | train_loss=-0.0116 | train_item=0.0206 | train_price=0.0276 | val_loss=-0.0305 | val_item=0.0148 | val_price=0.0194


epoch=9 | train_loss=-0.0326 | train_item=0.0152 | train_price=0.0244 | val_loss=-0.0463 | val_item=0.0122 | val_price=0.0178
epoch=10 | train_loss=-0.0478 | train_item=0.0130 | train_price=0.0211 | val_loss=-0.0582 | val_item=0.0104 | val_price=0.0165


epoch=11 | train_loss=-0.0594 | train_item=0.0116 | train_price=0.0176 | val_loss=-0.0665 | val_item=0.0095 | val_price=0.0154
epoch=12 | train_loss=-0.0651 | train_item=0.0111 | train_price=0.0183 | val_loss=-0.0714 | val_item=0.0092 | val_price=0.0147


epoch=13 | train_loss=-0.0681 | train_item=0.0107 | train_price=0.0196 | val_loss=-0.0745 | val_item=0.0089 | val_price=0.0137
epoch=14 | train_loss=-0.0724 | train_item=0.0103 | train_price=0.0156 | val_loss=-0.0758 | val_item=0.0088 | val_price=0.0131


epoch=15 | train_loss=-0.0716 | train_item=0.0101 | train_price=0.0180 | val_loss=-0.0760 | val_item=0.0088 | val_price=0.0131
best epoch: 15


## The history is a list of dicts, one per epoch

In [4]:
history = pd.DataFrame(model.history_).set_index("epoch")
history.round(4)

,train_loss,train_item,train_price,val_loss,val_item,val_price
epoch,,,,,,
1,2.4540,1.7857,1.3515,1.9034,1.5804,0.6857
2,1.5506,1.3007,0.5411,1.2984,1.0942,0.4572
3,0.9820,0.8183,0.3599,0.6405,0.5516,0.1947
4,0.4758,0.3964,0.1654,0.3478,0.2663,0.1653
5,0.2040,0.1676,0.0778,0.1084,0.0889,0.0534
6,0.0696,0.0611,0.0434,0.0273,0.0360,0.0283
7,0.0181,0.0320,0.0318,-0.0068,0.0231,0.0203
8,-0.0116,0.0206,0.0276,-0.0305,0.0148,0.0194
9,-0.0326,0.0152,0.0244,-0.0463,0.0122,0.0178


## Why the total loss goes negative

The tasks are balanced with learned uncertainty weighting
([Kendall et al., 2018](https://arxiv.org/abs/1705.07115)). Each task adds `0.5 · s`, where
`s = log σ²` is learned. As the model gets confident, `s` drops below zero and so does the total.
That is expected. **Compare the per-feature losses** (`train_item`, `val_price`, ...) instead.

The learned weights show how much each task counts:

In [5]:
model.task_weights()

{'item': 0.9976435899734497, 'price': 0.6032413840293884}

## A loader that doesn't match the model is rejected before training

In [6]:
mismatched = DataLoader(CoreDataset(train), batch_size=32,
                        collate_fn=CustomCollate(features, "item", funcs, max_len=16, recent_first=False))
try:
    NextSequence(config, epochs=1, verbose=False).fit(mismatched)
except ValueError as e:
    print("ValueError:", e)

ValueError: the train loader has recent_first=False but the model expects recent_first=True. Build it with CustomCollate.from_config(config, ...).


## Reproducibility: the same `seed` gives the same run

In [7]:
run_a = NextSequence(config, epochs=2, lr=3e-3, seed=42, verbose=False, device="cpu").fit(loader(train, shuffle=True))
run_b = NextSequence(config, epochs=2, lr=3e-3, seed=42, verbose=False, device="cpu").fit(loader(train, shuffle=True))
print("identical histories:", run_a.history_ == run_b.history_)

identical histories: True


## Continue training with `warm_start=True`

By default, calling `fit` again starts from a fresh model. With `warm_start=True` it continues from
the current weights, and the history keeps counting epochs.

In [8]:
more = NextSequence(config, epochs=3, lr=1e-3, seed=0, verbose=False, warm_start=True)
more.fit(loader(train, shuffle=True), loader(val))
more.fit(loader(train, shuffle=True), loader(val))
pd.DataFrame(more.history_)[["epoch", "train_loss", "val_loss"]].round(4)

,epoch,train_loss,val_loss
0,1,2.3793,2.0298
1,2,1.8926,1.8589
2,3,1.7603,1.8191
3,4,1.5902,1.4316
4,5,1.2438,1.2526
5,6,1.0970,1.2119
